[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/recsys-course/00_intro/00_proyecto_eda_popularidad.ipynb)

# 🛠️ Proyecto 00 · EDA de MovieLens y la primera home de CineMatch

| | |
|---|---|
| **Nivel** | 🟢 Básico |
| **Duración estimada** | 2 h |
| **GPU** | No necesaria. ≈ 0 unidades de Colab |
| **Prerrequisitos** | Lección 00 |

## 🎬 Contexto de negocio
Acabas de incorporarte como **primera ML engineer de CineMatch**, una plataforma de streaming que se lanza en dos semanas. La CEO quiere que la home **no sea un catálogo alfabético**. Tienes un volcado de valoraciones de una beta (usaremos **MovieLens-100K**) y te pide:

1. Un **análisis exploratorio** que el equipo de producto pueda entender (¿cuánto sabemos de cada usuario? ¿qué películas mueven la plataforma?).
2. Una **primera home**: fila "Top 10 en CineMatch" + filas por género.
3. Una **crítica honesta**: ¿qué tal funciona? ¿qué le falta? ¿qué harías en el siguiente sprint?

## 📦 Dataset
**MovieLens-100K** (GroupLens): 100.000 ratings (1–5) de 943 usuarios sobre 1.682 películas, 1997–1998, con géneros. Descarga oficial: <https://files.grouplens.org/datasets/movielens/ml-100k.zip>. Si no hay red, el cargador genera datos sintéticos con el mismo esquema.

## ✅ Entregables y rúbrica
| Entregable | Criterio |
|---|---|
| EDA (6 gráficos) | distribución de ratings, long tail con curva de Lorenz, actividad por usuario, ratings en el tiempo, géneros, nota media vs popularidad |
| 3 recomendadores no personalizados | popularidad, mejor nota media, **media bayesiana** (nota media "amortiguada") |
| Home por filas | `home(user_id)` devuelve un dict `fila → lista de títulos` sin repetir películas ni mostrar las ya vistas |
| Mini-evaluación temporal | HitRate@10 de cada recomendador ocultando las últimas 5 valoraciones (≥ 4★) de cada usuario. **Objetivo: HitRate@10 de la popularidad ≥ 3× el de "mejor nota media"** |
| Crítica | 5 limitaciones concretas con una propuesta para cada una |

In [ ]:
!pip install -q pandas matplotlib pyarrow

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

In [ ]:
#@title 🔧 Utilidades del curso: cinematch_data.py { display-mode: "form" }
# Escribe los módulos reutilizables del curso en el directorio actual.
# Última versión en GitHub (alternativa):
#   !wget -q https://raw.githubusercontent.com/padillacm/IA-practice/master/recsys-course/01_data/cinematch_data.py
_SOURCES = {}
_SOURCES['cinematch_data.py'] = '"""cinematch_data — pipeline de datos de CineMatch (módulo 01 del curso).\n\nInterfaz estable que reutilizan los módulos posteriores:\n\n    ratings, items = load_movielens("100k")      # o "1m", "latest-small", "25m", "32m"\n    users = load_users("100k")                    # demografía (solo 100k y 1m; si no, None)\n    inter = to_implicit(ratings, threshold=4.0)   # feedback implícito (rating >= 4)\n    inter = filter_k_core(inter, min_user=5, min_item=5)\n    train, val, test = temporal_split(inter, val_frac=0.1, test_frac=0.1)\n    train, test = leave_one_out_split(inter)      # último ítem de cada usuario a test\n    train, test = random_split(inter, test_frac=0.2)\n    X, u2i, i2i = to_csr(train)                   # matriz usuario×ítem scipy.sparse\n    recs = recommend_popular(train, users=test.user_id.unique(), k=10)\n    # recs: dict[user_id -> list[item_id]] ordenado de más a menos recomendado\n\nConvenciones de columnas: user_id (int), item_id (int), rating (float),\ntimestamp (int, segundos Unix). items: item_id, title, year, genres (str "A|B").\n\nSi GroupLens no responde, 100k y 1m se descargan de un espejo público en GitHub\n(mismo contenido); si tampoco hay red, `load_movielens` genera un dataset\nsintético con el mismo esquema (`make_synthetic_movielens`) y avisa.\n"""\nfrom __future__ import annotations\n\nimport io\nimport urllib.request\nimport warnings\nimport zipfile\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport scipy.sparse as sp\n\nBASE_URL = "https://files.grouplens.org/datasets/movielens"\nSIZES = {\n    "100k": "ml-100k",\n    "1m": "ml-1m",\n    "latest-small": "ml-latest-small",\n    "25m": "ml-25m",\n    "32m": "ml-32m",\n}\nGENRES_100K = [\n    "unknown", "Action", "Adventure", "Animation", "Children\'s", "Comedy", "Crime",\n    "Documentary", "Drama", "Fantasy", "Film-Noir", "Horror", "Musical", "Mystery",\n    "Romance", "Sci-Fi", "Thriller", "War", "Western",\n]\n\n\n# ----------------------------------------------------------------------------\n# Descarga y carga\n# ----------------------------------------------------------------------------\ndef download_movielens(size: str = "100k", data_dir: str | Path = "data") -> Path:\n    """Descarga y descomprime MovieLens desde GroupLens. Devuelve la carpeta."""\n    if size not in SIZES:\n        raise ValueError(f"size debe ser uno de {list(SIZES)}")\n    name = SIZES[size]\n    data_dir = Path(data_dir)\n    folder = data_dir / name\n    if folder.exists():\n        return folder\n    data_dir.mkdir(parents=True, exist_ok=True)\n    url = f"{BASE_URL}/{name}.zip"\n    print(f"Descargando {url} ...")\n    try:\n        with urllib.request.urlopen(url, timeout=60) as resp:\n            zipfile.ZipFile(io.BytesIO(resp.read())).extractall(data_dir)\n    except Exception as e:\n        if size not in MIRRORS:\n            raise\n        print(f"GroupLens no responde ({e.__class__.__name__}); probando espejo público en GitHub…")\n        _download_mirror(size, folder)\n    return folder\n\n\n# Espejos públicos (mismo contenido que GroupLens) por si files.grouplens.org no responde.\nMIRRORS = {\n    "100k": "https://raw.githubusercontent.com/RUCAIBox/RecBole/master/dataset/ml-100k",\n    "1m": "https://raw.githubusercontent.com/khanhnamle1994/movielens/master",\n}\n\n\ndef _write_dat(df: pd.DataFrame, path: Path) -> None:\n    """Escribe en el formato \'::\' de MovieLens-1M."""\n    lines = ("::".join(map(str, row)) for row in df.itertuples(index=False))\n    path.write_text("\\n".join(lines) + "\\n", encoding="latin-1", errors="replace")\n\n\ndef _download_mirror(size: str, folder: Path) -> None:\n    """Descarga el espejo y lo reescribe con los nombres y el formato originales de\n    GroupLens, para que `_read_folder` y `load_users` funcionen sin cambios."""\n    import csv\n    base, tmp = MIRRORS[size], folder.with_name(folder.name + ".tmp")\n    tmp.mkdir(parents=True, exist_ok=True)\n    if size == "100k":\n        rd = lambda f: pd.read_csv(f"{base}/ml-100k.{f}", sep="\\t", quoting=csv.QUOTE_NONE)\n        inter, it, us = rd("inter"), rd("item"), rd("user")\n        inter.columns, it.columns = ["u", "i", "r", "t"], ["item_id", "title", "year", "genres"]\n        inter.astype("int64").to_csv(tmp / "u.data", sep="\\t", header=False, index=False)\n        year = pd.to_numeric(it["year"], errors="coerce")\n        title = [f"{t} ({int(y)})" if y == y else t for t, y in zip(it["title"], year)]\n        flags = pd.DataFrame({g: it["genres"].str.split(" ").apply(lambda gs: int(g in gs))\n                              for g in GENRES_100K})\n        u_item = pd.concat([it[["item_id"]], pd.Series(title, name="title"),\n                            pd.DataFrame({"rd": "", "vrd": "", "url": ""}, index=it.index), flags], axis=1)\n        u_item.to_csv(tmp / "u.item", sep="|", header=False, index=False, encoding="latin-1",\n                      errors="replace", quoting=csv.QUOTE_NONE, escapechar="\\\\")\n        us.to_csv(tmp / "u.user", sep="|", header=False, index=False)\n    else:  # 1m\n        rd = lambda f: pd.read_csv(f"{base}/{f}.csv", sep="\\t", index_col=0, encoding="latin-1")\n        _write_dat(rd("ratings")[["user_id", "movie_id", "rating", "timestamp"]], tmp / "ratings.dat")\n        _write_dat(rd("movies")[["movie_id", "title", "genres"]], tmp / "movies.dat")\n        _write_dat(rd("users")[["user_id", "gender", "age", "occupation", "zipcode"]], tmp / "users.dat")\n    tmp.rename(folder)\n\n\ndef _split_title_year(titles: pd.Series) -> tuple[pd.Series, pd.Series]:\n    year = titles.str.extract(r"\\((\\d{4})\\)\\s*$")[0].astype("float")\n    clean = titles.str.replace(r"\\s*\\(\\d{4}\\)\\s*$", "", regex=True)\n    return clean, year\n\n\ndef _read_folder(size: str, folder: Path) -> tuple[pd.DataFrame, pd.DataFrame]:\n    if size == "100k":\n        ratings = pd.read_csv(folder / "u.data", sep="\\t",\n                              names=["user_id", "item_id", "rating", "timestamp"])\n        cols = ["item_id", "title", "release_date", "video_release_date", "url"] + GENRES_100K\n        raw = pd.read_csv(folder / "u.item", sep="|", names=cols, encoding="latin-1")\n        flags = raw[GENRES_100K].values.astype(bool)\n        genres = ["|".join(g for g, f in zip(GENRES_100K, row) if f) or "unknown" for row in flags]\n        items = pd.DataFrame({"item_id": raw["item_id"], "title": raw["title"].fillna(""),\n                              "genres": genres})\n    elif size == "1m":\n        ratings = pd.read_csv(folder / "ratings.dat", sep="::", engine="python",\n                              names=["user_id", "item_id", "rating", "timestamp"])\n        items = pd.read_csv(folder / "movies.dat", sep="::", engine="python",\n                            names=["item_id", "title", "genres"], encoding="latin-1")\n    else:\n        ratings = pd.read_csv(folder / "ratings.csv").rename(\n            columns={"userId": "user_id", "movieId": "item_id"})\n        items = pd.read_csv(folder / "movies.csv").rename(columns={"movieId": "item_id"})\n    items["title"], items["year"] = _split_title_year(items["title"].astype(str))\n    return ratings, items[["item_id", "title", "year", "genres"]]\n\n\ndef load_movielens(size: str = "100k", data_dir: str | Path = "data",\n                   synthetic_fallback: bool = True) -> tuple[pd.DataFrame, pd.DataFrame]:\n    """Devuelve (ratings, items) con el esquema estándar de CineMatch.\n\n    ratings: user_id, item_id, rating (float32), timestamp (int64), ordenado por tiempo.\n    items:   item_id, title, year, genres ("Action|Comedy").\n    """\n    try:\n        folder = download_movielens(size, data_dir)\n        ratings, items = _read_folder(size, folder)\n    except Exception as e:  # sin red, URL caída, etc.\n        if not synthetic_fallback:\n            raise\n        warnings.warn(f"No se pudo descargar MovieLens-{size} ({e!r}). "\n                      "Usando datos SINTÉTICOS con el mismo esquema.")\n        ratings, items = make_synthetic_movielens()\n    ratings = ratings.astype({"user_id": "int64", "item_id": "int64",\n                              "rating": "float32", "timestamp": "int64"})\n    ratings = ratings.sort_values(["timestamp", "user_id", "item_id"], kind="stable")\n    return ratings.reset_index(drop=True), items.reset_index(drop=True)\n\n\ndef load_users(size: str = "100k", data_dir: str | Path = "data") -> pd.DataFrame | None:\n    """Demografía de usuarios (user_id, age, gender, occupation). Solo 100k y 1m."""\n    try:\n        folder = download_movielens(size, data_dir)\n    except Exception:\n        return None\n    if size == "100k":\n        return pd.read_csv(folder / "u.user", sep="|",\n                           names=["user_id", "age", "gender", "occupation", "zip"])\n    if size == "1m":\n        return pd.read_csv(folder / "users.dat", sep="::", engine="python",\n                           names=["user_id", "gender", "age", "occupation", "zip"])\n    return None\n\n\ndef make_synthetic_movielens(n_users: int = 943, n_items: int = 1682,\n                             n_ratings: int = 100_000, seed: int = 42\n                             ) -> tuple[pd.DataFrame, pd.DataFrame]:\n    """Dataset sintético con forma de MovieLens: long tail, actividad sesgada,\n    gustos latentes por género y deriva temporal (los ítems nuevos van llegando)."""\n    rng = np.random.default_rng(seed)\n    genres = GENRES_100K[1:]\n    t0, t1 = 874_724_710, 893_286_638  # mismo rango temporal que ML-100K\n    # Ítems: género principal, popularidad intrínseca (Zipf) y fecha de llegada\n    item_genre = rng.integers(0, len(genres), n_items)\n    item_pop = 1.0 / np.arange(1, n_items + 1) ** 1.1\n    rng.shuffle(item_pop)\n    item_arrival = t0 + (t1 - t0) * np.where(rng.random(n_items) < 0.5, 0.0,\n                                             rng.uniform(0, 0.95, n_items))  # catálogo inicial + estrenos\n    # Usuarios: actividad log-normal, afinidad por géneros y fecha de alta\n    activity = rng.lognormal(0, 1.0, n_users)\n    activity = np.maximum(20, activity / activity.sum() * n_ratings).astype(int)\n    taste = rng.dirichlet(np.full(len(genres), 0.3), n_users)\n    user_start = t0 + (t1 - t0) * rng.uniform(0, 0.9, n_users)\n    item_quality = rng.normal(0, 0.45, n_items)   # unas películas son mejores que otras\n    user_bias = rng.normal(0, 0.35, n_users)      # hay usuarios generosos y exigentes\n    rows = []\n    for u in range(n_users):\n        start = user_start[u]\n        end = min(t1, start + rng.uniform(1, 120) * 86_400)\n        avail = item_arrival <= end\n        n = int(min(activity[u], avail.sum() // 2))\n        ts = np.sort(rng.uniform(start, end, n))\n        age_days = np.maximum(0, end - item_arrival) / 86_400\n        hype = np.where(item_arrival > t0, 1 + 30 * np.exp(-age_days / 30), 1.0)  # los estrenos tienen su momento\n        w = item_pop * hype * (0.15 + taste[u][item_genre]) * avail\n        chosen = rng.choice(n_items, size=n, replace=False, p=w / w.sum())\n        ts = np.maximum(ts, item_arrival[chosen] + 3600)\n        aff = taste[u][item_genre[chosen]] * len(genres)\n        r = np.clip(np.round(3.2 + 0.6 * np.log1p(aff) + item_quality[chosen] + user_bias[u]\n                             + rng.normal(0, 0.7, n)), 1, 5)\n        rows.append(pd.DataFrame({"user_id": u + 1, "item_id": chosen + 1,\n                                  "rating": r, "timestamp": ts.astype(np.int64)}))\n    ratings = pd.concat(rows, ignore_index=True)\n    second = rng.integers(0, len(genres), n_items)\n    items = pd.DataFrame({\n        "item_id": np.arange(1, n_items + 1),\n        "title": [f"Película sintética {i}" for i in range(1, n_items + 1)],\n        "year": (1930 + 66 * rng.beta(4, 1.5, n_items)).round(),\n        "genres": [genres[a] if a == b else f"{genres[a]}|{genres[b]}"\n                   for a, b in zip(item_genre, second)],\n    })\n    return ratings, items\n\n\n# ----------------------------------------------------------------------------\n# Limpieza y señales\n# ----------------------------------------------------------------------------\ndef to_implicit(ratings: pd.DataFrame, threshold: float = 4.0) -> pd.DataFrame:\n    """Convierte ratings explícitos en interacciones implícitas positivas\n    (rating >= threshold). Mantiene la columna rating por si se quiere ponderar."""\n    out = ratings[ratings["rating"] >= threshold].copy()\n    return out.reset_index(drop=True)\n\n\ndef dedup_interactions(df: pd.DataFrame, keep: str = "last") -> pd.DataFrame:\n    """Elimina pares (usuario, ítem) duplicados quedándose con el primero/último en el tiempo."""\n    return (df.sort_values("timestamp", kind="stable")\n              .drop_duplicates(["user_id", "item_id"], keep=keep)\n              .reset_index(drop=True))\n\n\ndef filter_k_core(df: pd.DataFrame, min_user: int = 5, min_item: int = 5,\n                  max_iter: int = 100) -> pd.DataFrame:\n    """k-core iterativo: repite hasta que todo usuario tenga >= min_user\n    interacciones y todo ítem >= min_item (filtrar uno puede romper el otro)."""\n    for _ in range(max_iter):\n        n0 = len(df)\n        ic = df["item_id"].map(df["item_id"].value_counts())\n        df = df[ic >= min_item]\n        uc = df["user_id"].map(df["user_id"].value_counts())\n        df = df[uc >= min_user]\n        if len(df) == n0:\n            break\n    return df.reset_index(drop=True)\n\n\n# ----------------------------------------------------------------------------\n# Splits\n# ----------------------------------------------------------------------------\ndef _drop_cold(train: pd.DataFrame, other: pd.DataFrame, users: bool, items: bool) -> pd.DataFrame:\n    if users:\n        other = other[other["user_id"].isin(train["user_id"].unique())]\n    if items:\n        other = other[other["item_id"].isin(train["item_id"].unique())]\n    return other.reset_index(drop=True)\n\n\ndef random_split(df: pd.DataFrame, test_frac: float = 0.2, seed: int = 42,\n                 drop_cold: bool = True) -> tuple[pd.DataFrame, pd.DataFrame]:\n    """Split aleatorio de interacciones (¡filtra el futuro al train! solo como contraejemplo)."""\n    mask = np.random.default_rng(seed).random(len(df)) < test_frac\n    train, test = df[~mask].reset_index(drop=True), df[mask]\n    return train, _drop_cold(train, test, drop_cold, drop_cold)\n\n\ndef leave_one_out_split(df: pd.DataFrame, n_test: int = 1, min_train: int = 1\n                        ) -> tuple[pd.DataFrame, pd.DataFrame]:\n    """Las últimas n_test interacciones (en el tiempo) de cada usuario van a test.\n    Ojo: no hay un corte temporal global → el train de un usuario puede contener\n    interacciones posteriores al test de otro (leakage temporal entre usuarios)."""\n    df = df.sort_values(["user_id", "timestamp"], kind="stable")\n    rank_from_end = df.groupby("user_id").cumcount(ascending=False)\n    n_user = df.groupby("user_id")["item_id"].transform("size")\n    is_test = (rank_from_end < n_test) & (n_user >= n_test + min_train)\n    return df[~is_test].reset_index(drop=True), df[is_test].reset_index(drop=True)\n\n\ndef temporal_split(df: pd.DataFrame, val_frac: float = 0.1, test_frac: float = 0.1,\n                   drop_cold_users: bool = True, drop_cold_items: bool = True\n                   ) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:\n    """Split temporal GLOBAL: train = pasado, val = siguiente val_frac del tiempo\n    (por cuantiles de eventos), test = último test_frac. Es lo más parecido a\n    producción: entrenas con todo lo anterior a T y sirves después de T.\n    Devuelve (train, val, test); val puede estar vacío si val_frac == 0."""\n    ts = df["timestamp"]\n    t_test = ts.quantile(1 - test_frac)\n    t_val = ts.quantile(1 - test_frac - val_frac) if val_frac > 0 else t_test\n    train = df[ts < t_val].reset_index(drop=True)\n    val = df[(ts >= t_val) & (ts < t_test)]\n    test = df[ts >= t_test]\n    val = _drop_cold(train, val, drop_cold_users, drop_cold_items)\n    # Para test, "conocido" = visto en train o val (en producción reentrenas antes de servir)\n    seen = pd.concat([train, val])\n    test = _drop_cold(seen, test, drop_cold_users, drop_cold_items)\n    return train, val, test\n\n\n# ----------------------------------------------------------------------------\n# Matrices dispersas\n# ----------------------------------------------------------------------------\ndef build_mappings(df: pd.DataFrame) -> tuple[dict, dict]:\n    """Mapeos id original -> índice contiguo 0..n-1 (usuarios e ítems)."""\n    users = np.sort(df["user_id"].unique())\n    items = np.sort(df["item_id"].unique())\n    return ({u: i for i, u in enumerate(users)}, {it: j for j, it in enumerate(items)})\n\n\ndef to_csr(df: pd.DataFrame, user2idx: dict | None = None, item2idx: dict | None = None,\n           value_col: str | None = None) -> tuple[sp.csr_matrix, dict, dict]:\n    """Matriz usuario×ítem CSR. value_col=None → 1.0 por interacción (implícito).\n    Las filas/ítems fuera del mapeo se descartan. Devuelve (X, user2idx, item2idx)."""\n    if user2idx is None or item2idx is None:\n        user2idx, item2idx = build_mappings(df)\n    r = df["user_id"].map(user2idx)\n    c = df["item_id"].map(item2idx)\n    ok = r.notna() & c.notna()\n    vals = np.ones(ok.sum(), dtype=np.float32) if value_col is None \\\n        else df.loc[ok, value_col].to_numpy(np.float32)\n    X = sp.csr_matrix((vals, (r[ok].astype(int), c[ok].astype(int))),\n                      shape=(len(user2idx), len(item2idx)))\n    X.sum_duplicates()\n    return X, user2idx, item2idx\n\n\n# ----------------------------------------------------------------------------\n# Baselines\n# ----------------------------------------------------------------------------\ndef _seen_by_user(train: pd.DataFrame) -> dict:\n    return train.groupby("user_id")["item_id"].agg(set).to_dict()\n\n\ndef _topk_excluding(ranked: np.ndarray, seen: set, k: int) -> list:\n    out = []\n    for it in ranked:\n        if it not in seen:\n            out.append(int(it))\n            if len(out) == k:\n                break\n    return out\n\n\ndef popularity_ranking(train: pd.DataFrame, window_days: float | None = None,\n                       half_life_days: float | None = None) -> np.ndarray:\n    """Ítems ordenados por popularidad. window_days: solo cuenta los últimos N días.\n    half_life_days: pondera cada evento por 0.5**(edad/half_life) (recencia suave)."""\n    df = train\n    t_max = df["timestamp"].max()\n    if window_days is not None:\n        df = df[df["timestamp"] >= t_max - window_days * 86_400]\n    if half_life_days is not None:\n        age = (t_max - df["timestamp"]) / 86_400\n        w = 0.5 ** (age / half_life_days)\n        scores = w.groupby(df["item_id"]).sum()\n    else:\n        scores = df["item_id"].value_counts()\n    return scores.sort_values(ascending=False, kind="stable").index.to_numpy()\n\n\ndef recommend_popular(train: pd.DataFrame, users, k: int = 10, exclude_seen: bool = True,\n                      window_days: float | None = None,\n                      half_life_days: float | None = None) -> dict:\n    """Top-K más populares (opcionalmente recientes) para cada usuario de `users`."""\n    ranked = popularity_ranking(train, window_days, half_life_days)\n    seen = _seen_by_user(train) if exclude_seen else {}\n    return {u: _topk_excluding(ranked, seen.get(u, set()), k) for u in users}\n\n\ndef recommend_popular_by_segment(train: pd.DataFrame, users, segment_of: dict, k: int = 10,\n                                 exclude_seen: bool = True, min_segment_events: int = 50) -> dict:\n    """Popularidad dentro del segmento del usuario (p. ej. edad, país, género favorito).\n    `segment_of`: dict user_id -> segmento. Segmentos con pocos eventos → popularidad global."""\n    seg = train["user_id"].map(segment_of)\n    global_rank = popularity_ranking(train)\n    ranks = {}\n    for s, g in train.groupby(seg):\n        if len(g) >= min_segment_events:\n            r = g["item_id"].value_counts().index.to_numpy()\n            # completar con el ranking global para no quedarnos cortos\n            ranks[s] = np.concatenate([r, global_rank[~np.isin(global_rank, r)]])\n    seen = _seen_by_user(train) if exclude_seen else {}\n    return {u: _topk_excluding(ranks.get(segment_of.get(u), global_rank), seen.get(u, set()), k)\n            for u in users}\n\n\ndef recommend_random(train: pd.DataFrame, users, k: int = 10, seed: int = 42,\n                     exclude_seen: bool = True) -> dict:\n    """Recomendador aleatorio: el suelo absoluto de cualquier comparación."""\n    rng = np.random.default_rng(seed)\n    items = train["item_id"].unique()\n    seen = _seen_by_user(train) if exclude_seen else {}\n    return {u: _topk_excluding(rng.permutation(items), seen.get(u, set()), k) for u in users}\n\n\n# ----------------------------------------------------------------------------\n# Persistencia\n# ----------------------------------------------------------------------------\ndef save_splits(out_dir: str | Path, **frames: pd.DataFrame) -> Path:\n    """Guarda cada DataFrame como parquet: save_splits("data/cinematch", train=..., test=...)."""\n    out_dir = Path(out_dir)\n    out_dir.mkdir(parents=True, exist_ok=True)\n    for name, f in frames.items():\n        f.to_parquet(out_dir / f"{name}.parquet", index=False)\n    return out_dir\n\n\ndef load_splits(out_dir: str | Path, names=("train", "val", "test", "items")) -> dict:\n    out_dir = Path(out_dir)\n    return {n: pd.read_parquet(out_dir / f"{n}.parquet") for n in names\n            if (out_dir / f"{n}.parquet").exists()}\n\n\n# ----------------------------------------------------------------------------\n# Pipeline completo de CineMatch (una llamada)\n# ----------------------------------------------------------------------------\ndef prepare_cinematch(size: str = "100k", threshold: float | None = 4.0, min_user: int = 5,\n                      min_item: int = 5, val_frac: float = 0.1, test_frac: float = 0.1,\n                      data_dir: str | Path = "data", out_dir: str | Path | None = None) -> dict:\n    """Pipeline estándar del curso:\n    carga → dedup → implícito (rating >= threshold; None = no filtrar) → split temporal\n    global → k-core SOLO sobre train (no miramos el futuro para filtrar) → val/test\n    restringidos a usuarios e ítems conocidos en train.\n\n    Devuelve {"train", "val", "test", "items", "ratings"} (DataFrames) y, si\n    out_dir no es None, los guarda en parquet."""\n    ratings, items = load_movielens(size, data_dir)\n    inter = dedup_interactions(ratings)\n    if threshold is not None:\n        inter = to_implicit(inter, threshold)\n    train, val, test = temporal_split(inter, val_frac, test_frac,\n                                      drop_cold_users=False, drop_cold_items=False)\n    train = filter_k_core(train, min_user, min_item)\n    val = _drop_cold(train, val, True, True)\n    test = _drop_cold(train, test, True, True)\n    out = {"train": train, "val": val, "test": test, "items": items, "ratings": ratings}\n    if out_dir is not None:\n        save_splits(out_dir, train=train, val=val, test=test, items=items)\n    return out\n'
for _name, _src in _SOURCES.items():
    with open(_name, 'w', encoding='utf-8') as _fh:
        _fh.write(_src)
    print('✔ escrito', _name)

In [ ]:
from cinematch_data import load_movielens

ratings, items = load_movielens("100k")
df = ratings.merge(items, on="item_id")
df["fecha"] = pd.to_datetime(df.timestamp, unit="s")
df.head()

---
## Parte 1 · EDA

### TODO 1 — Cifras clave
Calcula y muestra: nº de usuarios, películas, ratings, **densidad** de la matriz ($|\text{ratings}| / (|U|\cdot|I|)$), rating medio, mediana de ratings por usuario y por película, y % de películas con menos de 5 ratings.

In [ ]:
def cifras_clave(df: pd.DataFrame) -> pd.Series:
    # TODO: devuelve una pd.Series con las cifras pedidas (nombres en español)
    raise NotImplementedError

cifras_clave(df)

### TODO 2 — Seis gráficos
1. Distribución de ratings (barras).
2. **Curva de Lorenz** de la popularidad de películas (eje x: % de películas de menos a más populares; eje y: % acumulado de ratings) + índice de Gini.
3. Histograma de ratings por usuario (escala log).
4. Nº de ratings por semana a lo largo del tiempo.
5. Nº de películas y nº de ratings por género (un ítem con varios géneros cuenta en cada uno; pista: `str.split("|")` + `explode`).
6. Dispersión **nota media vs nº de ratings** por película (eje x log). ¿Qué ves en la zona de pocos ratings?

> 💡 Pista Gini: con las popularidades ordenadas de menor a mayor $x_{(1)} \le \dots \le x_{(n)}$, $G = \frac{\sum_{k=1}^{n} (2k - n - 1)\, x_{(k)}}{n \sum_k x_{(k)}}$.

In [ ]:
def gini(x: np.ndarray) -> float:
    # TODO
    raise NotImplementedError

def graficos_eda(df: pd.DataFrame) -> None:
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    # TODO: los 6 gráficos, uno en cada eje de `axes.ravel()`, con títulos y ejes en español
    raise NotImplementedError

graficos_eda(df)

---
## Parte 2 · Tres recomendadores no personalizados

### TODO 3 — Popularidad, nota media y media bayesiana
La nota media "cruda" premia películas con **un solo** rating de 5★. La **media bayesiana** (la que usaba el Top 250 de IMDb) mezcla la media de la película con la media global $\mu$, con un peso $m$ (nº de "votos virtuales"):

$$\text{WR}_i = \frac{n_i}{n_i + m}\,\bar r_i + \frac{m}{n_i + m}\,\mu$$

Implementa tres funciones que devuelvan una `pd.Series` `item_id → puntuación` ordenada de mayor a menor. Prueba $m$ = percentil 75 del nº de ratings por película (pista: en ML-100K sale $m = 80$).

In [ ]:
def score_popularidad(train: pd.DataFrame) -> pd.Series:
    # TODO: nº de ratings por película, de mayor a menor
    raise NotImplementedError

def score_media(train: pd.DataFrame) -> pd.Series:
    # TODO: nota media por película, de mayor a menor
    raise NotImplementedError

def score_bayesiano(train: pd.DataFrame, m: float | None = None) -> pd.Series:
    # TODO: media bayesiana; si m es None usa el percentil 75 de n_i
    raise NotImplementedError

for nombre, f in [("popularidad", score_popularidad), ("media", score_media), ("bayesiana", score_bayesiano)]:
    top = f(df).head(5).index
    print(f"{nombre:12s}", items.set_index("item_id").loc[top, "title"].tolist())

### TODO 4 — La home por filas
Implementa `home(user_id, train, score_fn, generos, k)` que devuelva un `dict` con:
- `"Top 10 en CineMatch"`: los $k$ mejores según `score_fn`, excluyendo lo ya visto.
- Una fila por cada género de `generos` (los $k$ mejores de ese género según la misma puntuación).
- **Sin repetir** películas entre filas (la primera fila que la reclama se la queda) y **sin mostrar lo ya visto**.

In [ ]:
GENEROS_HOME = ["Comedy", "Drama", "Action", "Thriller", "Sci-Fi"]

def home(user_id: int, train: pd.DataFrame, score_fn=None, generos=GENEROS_HOME, k: int = 10) -> dict:
    # TODO
    raise NotImplementedError

for fila, titulos in home(1, df, score_bayesiano).items():
    print(f"{fila}: {titulos[:4]} ...")

---
## Parte 3 · ¿Funciona? Mini-evaluación temporal

Simulamos el futuro: para cada usuario ocultamos sus **5 últimas** valoraciones en el tiempo y nos quedamos como "relevantes" las que tienen ≥ 4★. Entrenamos con el resto y medimos **HitRate@10**: fracción de usuarios con al menos un relevante en su top-10 (excluyendo lo ya visto en train). En el módulo 02 verás métricas más finas.

### TODO 5 — Split y HitRate@10

In [ ]:
def split_ultimas(df: pd.DataFrame, n: int = 5) -> tuple[pd.DataFrame, pd.DataFrame]:
    # TODO: las n últimas interacciones de cada usuario (por timestamp) a test
    raise NotImplementedError

def hit_rate_at_k(score: pd.Series, train: pd.DataFrame, test: pd.DataFrame, k: int = 10) -> float:
    # TODO: relevantes = test con rating >= 4; recomienda top-k no vistos en train
    raise NotImplementedError

train, test = split_ultimas(df)
resultados = {n: hit_rate_at_k(f(train), train, test) for n, f in
              [("popularidad", score_popularidad), ("media", score_media), ("bayesiana", score_bayesiano)]}
resultados

### TODO 6 — Crítica (markdown)
Escribe 5 limitaciones concretas de esta home y una propuesta para cada una (piensa en: personalización, cobertura/long tail, tiempo/frescura, usuarios nuevos, bucle de retroalimentación, métricas).

*Tu respuesta aquí.*

---
## ⛔ SPOILER — Solución de referencia

Intenta resolverlo primero. Lo que sigue es una solución completa y ejecutable.

In [ ]:
def cifras_clave(df: pd.DataFrame) -> pd.Series:
    n_u, n_i, n_r = df.user_id.nunique(), df.item_id.nunique(), len(df)
    por_item = df.item_id.value_counts()
    return pd.Series({
        "usuarios": n_u, "películas": n_i, "ratings": n_r,
        "densidad": round(n_r / (n_u * n_i), 4),
        "rating medio": round(df.rating.mean(), 3),
        "mediana ratings/usuario": df.user_id.value_counts().median(),
        "mediana ratings/película": por_item.median(),
        "% películas con <5 ratings": round(100 * (por_item < 5).mean(), 1),
    })

cifras_clave(df)

In [ ]:
def gini(x: np.ndarray) -> float:
    x = np.sort(np.asarray(x, float))
    n = len(x)
    return float((2 * np.arange(1, n + 1) - n - 1) @ x / (n * x.sum()))

def graficos_eda(df: pd.DataFrame) -> None:
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    a = axes.ravel()
    df.rating.value_counts().sort_index().plot.bar(ax=a[0], color="#4C72B0", rot=0)
    a[0].set(title="1 · Distribución de ratings", xlabel="estrellas", ylabel="nº ratings")
    pop = np.sort(df.item_id.value_counts().values)
    a[1].plot(np.linspace(0, 100, len(pop)), 100 * np.cumsum(pop) / pop.sum(), color="#DD8452", label="Lorenz")
    a[1].plot([0, 100], [0, 100], "--", color="gray", label="igualdad perfecta")
    a[1].set(title=f"2 · Curva de Lorenz (Gini = {gini(pop):.2f})", xlabel="% películas (menos → más populares)",
             ylabel="% acumulado de ratings"); a[1].legend()
    a[2].hist(df.user_id.value_counts().values, bins=50, color="#55A868")
    a[2].set(title="3 · Ratings por usuario", xlabel="nº ratings", ylabel="nº usuarios", yscale="log")
    df.set_index("fecha").rating.resample("W").size().plot(ax=a[3], color="#8172B3")
    a[3].set(title="4 · Ratings por semana", xlabel="fecha", ylabel="nº ratings")
    g = df[["item_id", "genres"]].assign(genero=df.genres.str.split("|")).explode("genero")
    resumen = g.groupby("genero").agg(ratings=("item_id", "size"), peliculas=("item_id", "nunique"))
    resumen.sort_values("ratings").plot.barh(ax=a[4], color=["#4C72B0", "#DD8452"])
    a[4].legend(loc="lower right")
    a[4].set(title="5 · Ratings y películas por género", xlabel="cantidad", ylabel=""); a[4].set_xscale("log")
    st = df.groupby("item_id").rating.agg(["mean", "size"])
    a[5].scatter(st["size"], st["mean"], s=6, alpha=0.4, color="#C44E52")
    a[5].set(xscale="log", title="6 · Nota media vs popularidad", xlabel="nº ratings (log)", ylabel="nota media")
    plt.tight_layout(); plt.show()

graficos_eda(df)

**Lectura del gráfico 6**: a la izquierda (pocos ratings) la nota media se dispersa entre 1 y 5 — es **ruido**: una película con un único 5★ no es "la mejor del catálogo". A la derecha la varianza se estrecha. Es la razón de la media bayesiana.

In [ ]:
def score_popularidad(train: pd.DataFrame) -> pd.Series:
    return train.item_id.value_counts().sort_values(ascending=False, kind="stable")

def score_media(train: pd.DataFrame) -> pd.Series:
    return train.groupby("item_id").rating.mean().sort_values(ascending=False, kind="stable")

def score_bayesiano(train: pd.DataFrame, m: float | None = None) -> pd.Series:
    st = train.groupby("item_id").rating.agg(["mean", "size"])
    m = st["size"].quantile(0.75) if m is None else m
    mu = train.rating.mean()
    wr = st["size"] / (st["size"] + m) * st["mean"] + m / (st["size"] + m) * mu
    return wr.sort_values(ascending=False, kind="stable")

for nombre, f in [("popularidad", score_popularidad), ("media", score_media), ("bayesiana", score_bayesiano)]:
    top = f(df).head(5).index
    print(f"{nombre:12s}", items.set_index("item_id").loc[top, "title"].tolist())

In [ ]:
GENEROS_HOME = ["Comedy", "Drama", "Action", "Thriller", "Sci-Fi"]
titulo = items.set_index("item_id")["title"]
generos_de = items.set_index("item_id")["genres"].str.split("|")

def home(user_id: int, train: pd.DataFrame, score_fn=score_popularidad, generos=GENEROS_HOME, k: int = 10) -> dict:
    score = score_fn(train)
    vistas = set(train.loc[train.user_id == user_id, "item_id"])
    usadas: set = set()
    filas = {}
    candidatos = [i for i in score.index if i not in vistas]
    def llenar(pool):
        sel = [i for i in pool if i not in usadas][:k]
        usadas.update(sel)
        return [titulo[i] for i in sel]
    filas["Top 10 en CineMatch"] = llenar(candidatos)
    for g in generos:
        filas[f"Lo mejor de {g}"] = llenar([i for i in candidatos if g in generos_de.get(i, [])])
    return filas

for fila, titulos in home(1, df, score_bayesiano).items():
    print(f"{fila}: {titulos[:4]} ...")

In [ ]:
def split_ultimas(df: pd.DataFrame, n: int = 5) -> tuple[pd.DataFrame, pd.DataFrame]:
    df = df.sort_values(["user_id", "timestamp"], kind="stable")
    desde_final = df.groupby("user_id").cumcount(ascending=False)
    return df[desde_final >= n], df[desde_final < n]

def hit_rate_at_k(score: pd.Series, train: pd.DataFrame, test: pd.DataFrame, k: int = 10) -> float:
    relevantes = test[test.rating >= 4].groupby("user_id").item_id.agg(set)
    vistas = train.groupby("user_id").item_id.agg(set)
    orden = score.index.to_numpy()
    hits = []
    for u, rel in relevantes.items():
        vis = vistas.get(u, set())
        top = [i for i in orden[: k + len(vis)] if i not in vis][:k]
        hits.append(len(rel.intersection(top)) > 0)
    return float(np.mean(hits))

train, test = split_ultimas(df)
resultados = {n: hit_rate_at_k(f(train), train, test) for n, f in
              [("popularidad", score_popularidad), ("media", score_media), ("bayesiana", score_bayesiano)]}
pd.Series(resultados).plot.bar(color=["#4C72B0", "#C44E52", "#55A868"], rot=0,
                               title="HitRate@10 (últimas 5 valoraciones de cada usuario)")
plt.ylabel("HitRate@10"); plt.show()
resultados

In [ ]:
# Cobertura: ¿qué fracción del catálogo llega a enseñarse a algún usuario?
def cobertura(score: pd.Series, train: pd.DataFrame, k: int = 10) -> float:
    vistas = train.groupby("user_id").item_id.agg(set)
    orden = score.index.to_numpy()
    mostradas = set()
    for u, vis in vistas.items():
        mostradas.update([i for i in orden[: k + len(vis)] if i not in vis][:k])
    return len(mostradas) / train.item_id.nunique()

{n: round(cobertura(f(train), train), 3) for n, f in
 [("popularidad", score_popularidad), ("media", score_media), ("bayesiana", score_bayesiano)]}

### Crítica de referencia
1. **No es personal**: todos los usuarios ven la misma home (salvo lo ya visto). → Siguiente sprint: item-kNN / EASE (módulo 04), "porque viste X".
2. **Cobertura mínima** (unas decenas de películas de 1.682): la cola larga nunca se descubre y los productores de nicho no reciben exposición. → Métricas de cobertura y re-ranking por diversidad (módulos 02 y 13).
3. **Ignora el tiempo**: la popularidad "de siempre" no es la de esta semana. → Popularidad con ventana o decaimiento (módulo 01).
4. **Bucle de retroalimentación**: lo que mostramos es lo que se ve y lo que se ve es lo que mostramos. → Loguear impresiones, reservar exploración (módulo 14).
5. **"Mejor nota media" es engañosa**: favorece películas con 1–2 ratings; la media bayesiana lo corrige pero sigue sin personalizar. Y **HitRate** con un único corte por usuario es una evaluación pobre: falta un split temporal global, NDCG, intervalos de confianza (módulos 01 y 02).

Observa el resultado clave: el recomendador "más inteligente" (nota media) **pierde por goleada** frente a la popularidad. La popularidad captura algo real: la probabilidad *a priori* de que alguien vea algo. Es el primer "secreto de la élite": **nunca subestimes a la popularidad.**

---
## 🚀 Retos extra
1. **Popularidad reciente**: puntúa con los ratings de los últimos 30 días del train o con decaimiento exponencial. ¿Mejora el HitRate@10?
2. **Popularidad por segmento**: usa `load_users("100k")` (edad, género, ocupación) y calcula popularidad dentro de cada grupo de edad. ¿Gana a la global?
3. **Filas personalizadas**: ordena las filas de género según el género favorito de cada usuario.
4. **Intervalos de confianza**: bootstrap sobre usuarios del HitRate@10. ¿Es significativa la diferencia entre popularidad y media bayesiana?

## 🤔 Reflexión (producción / MLOps)
- Si esta home se despliega mañana, ¿qué eventos necesitas **loguear** desde el día 1 para poder entrenar algo mejor dentro de un mes? (Pista: **impresiones**, no solo clics.)
- ¿Cómo detectarías que la fila "Top 10" se ha quedado obsoleta? ¿Con qué frecuencia la recalcularías?
- ¿Qué *fallback* sirve tu API si el futuro modelo personalizado no responde a tiempo?